<a href="https://colab.research.google.com/github/jeneferTodt/analise-preditiva-entregas/blob/feat%2Fpipeline-dados/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 00 — Introdução


## 01 — Imports

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime, timedelta



## 02 — Dataset

In [34]:
## O gerador abaixo cria propositalmente dados “sujos”, que servirão de matéria-prima para o requisito de limpeza. Use-o como está:


import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)

# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())


Dataset gerado com 200 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


In [35]:
#  EXPORTAÇÃO DO DATASET

df_bruto.to_csv(
    "vendas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset salvo como vendas.csv")

# Conferir se o arquivo pode ser lido novamente
df_verificacao = pd.read_csv("vendas.csv")

display(df_verificacao.head())
print("Dimensões do arquivo:", df_verificacao.shape)

Dataset salvo como vendas.csv


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25


Dimensões do arquivo: (200, 8)


In [36]:
from google.colab import files

files.download("vendas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3 - Inspeção

In [42]:
## Descrever a estrutura do DataFrame

def inspecionar_dados(df_bruto):
    """Exibe as informações estruturais do DataFrame."""

    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Dimensões: {df_bruto.shape}")
    print(f"\nColunas: {list(df_bruto.columns)}")
    print(f"\nTipos de dados:\n{df_bruto.dtypes}")
    print(f"\nValores ausentes por coluna:\n{df_bruto.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df_bruto.head()}")

    return df_bruto


In [43]:
inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Dimensões: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores ausentes por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monit

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
...,...,...,...,...,...,...,...,...
195,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
196,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
197,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
198,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


## 4 - Limpeza e preparação

In [44]:

def tratar_dados(df):
    """
    Trata o DataFrame de vendas.

    Retorna:
        df_limpo: DataFrame tratado.
        relatorio: dicionário com as contagens da limpeza.
    """

    # Criar uma cópia para preservar o DataFrame original
    df_limpo = df.copy()

    # Registrar a quantidade inicial de registros
    registros_iniciais = len(df_limpo)

    # Contabilizar conversões e alterações
    relatorio = {
        "registros_iniciais": registros_iniciais,
        "textos_padronizados": 0,
        "datas_invalidas_removidas": 0,
        "quantidade_convertida": 0,
        "preco_unitario_convertido": 0,
        "registros_sem_alteracao": 0,
        "registros_finais": 0
    }

    # 1. Remover espaços extras nas colunas de texto
    colunas_texto = df_limpo.select_dtypes(
        include=["object", "string"]
    ).columns

    for coluna in colunas_texto:
        valores_antes = df_limpo[coluna].copy()

        df_limpo[coluna] = df_limpo[coluna].str.strip()

        relatorio["textos_padronizados"] += int(
            (
                valores_antes.notna()
                & valores_antes.ne(df_limpo[coluna])
            ).sum()
        )

    # 2. Converter datas e remover registros inválidos
    datas_antes = df_limpo["data_venda"].copy()

    df_limpo["data_venda"] = pd.to_datetime(
        df_limpo["data_venda"],
        errors="coerce"
    )

    datas_invalidas = df_limpo["data_venda"].isna()

    relatorio["datas_invalidas_removidas"] = int(
        datas_invalidas.sum()
    )

    df_limpo = df_limpo.loc[~datas_invalidas].copy()

    # 3. Converter quantidade para inteiro
    quantidade_antes = df_limpo["quantidade"].copy()

    quantidade_convertida = pd.to_numeric(
        quantidade_antes,
        errors="coerce"
    )

    quantidade_valida = (
        quantidade_convertida.notna()
        & (quantidade_convertida % 1 == 0)
    )

    # Não descartar silenciosamente valores inválidos
    if not quantidade_valida.all():
        raise ValueError(
            "Existem valores inválidos ou não inteiros "
            "na coluna quantidade."
        )

    relatorio["quantidade_convertida"] = int(
        (
            quantidade_antes.astype("string")
            != quantidade_convertida.astype("string")
        ).sum()
    )

    df_limpo["quantidade"] = quantidade_convertida.astype(int)

    # 4. Converter preco_unitario para float
    preco_antes = df_limpo["preco_unitario"].copy()

    preco_convertido = pd.to_numeric(
        preco_antes,
        errors="coerce"
    )

    preco_valido = preco_convertido.notna()

    if not preco_valido.all():
        raise ValueError(
            "Existem valores inválidos na coluna preco_unitario."
        )

    relatorio["preco_unitario_convertido"] = int(
        (
            preco_antes.astype("string")
            != preco_convertido.astype("string")
        ).sum()
    )

    df_limpo["preco_unitario"] = preco_convertido.astype(float)

    # 5. Calcular registros sem alteração
    registros_removidos = registros_iniciais - len(df_limpo)

    registros_com_alteracao = (
        relatorio["textos_padronizados"] > 0
        or relatorio["quantidade_convertida"] > 0
        or relatorio["preco_unitario_convertido"] > 0
        or relatorio["datas_invalidas_removidas"] > 0
    )

    relatorio["registros_finais"] = len(df_limpo)

    # Relatório no console
    print("=== RELATÓRIO DE LIMPEZA ===")
    print(f"Registros iniciais: {registros_iniciais}")
    print(
        "Espaços removidos em valores de texto: "
        f"{relatorio['textos_padronizados']}"
    )
    print(
        "Datas inválidas removidas: "
        f"{relatorio['datas_invalidas_removidas']}"
    )
    print(
        "Valores convertidos em quantidade: "
        f"{relatorio['quantidade_convertida']}"
    )
    print(
        "Valores convertidos em preco_unitario: "
        f"{relatorio['preco_unitario_convertido']}"
    )
    print(f"Registros finais: {relatorio['registros_finais']}")

    return df_limpo, relatorio


In [45]:

df_limpo, relatorio = tratar_dados(df_bruto)

display(df_limpo.head())

print("\nTipos de dados após a limpeza:")
print(df_limpo.dtypes)


=== RELATÓRIO DE LIMPEZA ===
Registros iniciais: 200
Espaços removidos em valores de texto: 0
Datas inválidas removidas: 6
Valores convertidos em quantidade: 0
Valores convertidos em preco_unitario: 0
Registros finais: 194


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
5,6,2025-07-03,Cliente_023,Notebook,Computadores,Sudeste,2,3373.92



Tipos de dados após a limpeza:
id_venda                   int64
data_venda        datetime64[ns]
cliente                   object
produto                   object
categoria                 object
regiao                    object
quantidade                 int64
preco_unitario           float64
dtype: object


In [46]:

assert pd.api.types.is_datetime64_any_dtype(
    df_limpo["data_venda"]
)

assert pd.api.types.is_integer_dtype(
    df_limpo["quantidade"]
)

assert pd.api.types.is_float_dtype(
    df_limpo["preco_unitario"]
)

assert df_limpo["data_venda"].notna().all()

print("RF03: validações concluídas com sucesso!")


RF03: validações concluídas com sucesso!


## 5 - Análise exploratória

## 06 — Gráficos

## 07 — Execução e resultados